# 28 Sep
-converting the lotus json to csv


In [ ]:
import pandas as pd

df = pd.read_json('/home/school/masters/Scripts/lotus/lotus_compounds.json')
df.to_csv('/home/school/masters/Scripts/lotus/lotus_compounds.csv', index=False)


## code above didi not have the plants and I need to use that as exclusion criteria so I got perplexity to only jason me the compounds that have a plant listed as making it and no i need to make that a csv

In [3]:
import pandas as pd

df = pd.read_json('/home/school/masters/Scripts/lotus/lotus_compounds_with_plant_source.json')
df.to_csv('/home/school/masters/Scripts/lotus/lotus_compounds_with_plant_source.csv', index=False)

In [5]:
df_tot_lotus = pd.read_csv('/home/school/masters/Scripts/lotus/lotus_compounds.csv')
df_plant_lotus = pd.read_csv('/home/school/masters/Scripts/lotus/lotus_compounds_with_plant_source.csv')

tot = df_tot_lotus.shape[0]
plant = df_plant_lotus.shape[0]

print(f'Og lotus has {tot:,} compounds')
print(f'Plant lotus has {plant:,} compounds')
print(f'Percentage of plant lotus: {plant/tot*100:.2f}%')
print(f'The difference is {tot-plant:,}')

Og lotus has 276,518 compounds
Plant lotus has 192,297 compounds
Percentage of plant lotus: 69.54%
The difference is 84,221


## now for inchi deduplication

In [ ]:
import pandas as pd

df = pd.read_csv('/home/school/masters/Scripts/lotus/lotus_compounds_with_plant_source.csv')

df.drop(columns=['_id','smiles'], inplace=True)

def neat_list(values):
    if not values:
        return None
    values = [value.strip for value in values]
    seen = set()
    out_list = []
    for value in values:
        if value not in seen and pd.notna(value):
            seen.add(value)
            out_list.append(value)
    return "|".join(out_list)
    
            

df_new = df.groupby('standard_inchi',as_index=False,sort=False).agg({'lotus_id' : neat_list,
                                                                    'wikidata_id': neat_list,
                                                                    'annotation_level': neat_list,
                                                                    'np_likeness': neat_list,
                                                                    'collections': neat_list,
                                                                    'dois': neat_list,
                                                                    'synonyms': neat_list,
                                                                    'cas': neat_list,
                                                                    'coconut_id': neat_list,
                                                                    'cid': neat_list})


## So the Lotus NP people used COCONUT for their data as well so I am just going to quickly check if there are any novel compounds 

In [13]:
import pandas as pd

lotus = pd.read_csv('/home/school/masters/Scripts/lotus/lotus_compounds_with_plant_source.csv')
coconut = pd.read_csv('/home/school/masters/Scripts/coconut_full/full_coconut_for_DB.csv')

lotus_inchi = lotus['inchi']
coconut_inchi = coconut['inchi']  # not lotus['inchi']

l = set(lotus_inchi.dropna())
c = set(coconut_inchi.dropna())

if l == c:
    print('LOTUS and Coconut contain exactly the same InChIs.')
elif l.issubset(c):
    print('All LOTUS InChIs are already present in Coconut.')
else:
    missing_from_coconut = l - c
    print(f'{len(missing_from_coconut)} LOTUS InChIs are missing from Coconut.')

71667 LOTUS InChIs are missing from Coconut.


## removing all entries with nplikeness less than -1

In [3]:
import pandas as pd

lotus = pd.read_csv('/home/school/masters/Scripts/lotus/lotus_compounds_with_plant_source.csv')

og = lotus.shape[0]

lotus['npl_score'] = pd.to_numeric(lotus['npl_score'], errors='coerce')

lotus = lotus.loc[lotus['npl_score']>-1].copy()
new = lotus.shape[0]

lotus.to_csv('/home/school/masters/Scripts/lotus/lotus_compounds_with_plant_source.csv',index=False)

print(f'The old was {og:,} long and now after removing lower than -1 np likeness score it is {new:,}')


The old was 192,297 long and now after removing lower than -1 np likeness score it is 191,703
